# 05 - Baselines

**Projeto:** MPEE_2026 - Aprendizado Profundo
**Tarefa:** classificacao de familia de causa raiz (6 classes)
**Pergunta:** features de processo + grupo de componente sao suficientes para prever a familia de causa raiz?

## Estrutura

| Secao | Conteudo | Tempo |
|---|---|---|
| 0 | Configuracao e imports | instantaneo |
| 1 | Carregamento dos dados | instantaneo |
| 2 | Pre-processamento | instantaneo |
| 3 | DummyClassifier (piso) | ~1s |
| 4 | Regressao Logistica | ~30s |
| 5 | Random Forest | ~60s |
| 6 | HistGradientBoosting | ~90s |
| 7 | Tabela comparativa | instantaneo |
| 8 | Matrizes de confusao | ~5s |
| 9 | Permutation importance | ~30s |
| 10 | Persistencia | instantaneo |

Ambiente: Python 3.15, sklearn 1.9.1, 6 nucleos, `n_jobs=5`.


## Secao 0 - Configuracao

Imports, semente, caminhos e constantes.

In [ ]:
from pathlib import Path
from datetime import datetime
import json, pickle, time, warnings
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier
from sklearn.metrics import (
    f1_score, balanced_accuracy_score, accuracy_score,
    confusion_matrix, classification_report, top_k_accuracy_score
)
from sklearn.inspection import permutation_importance

warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=UserWarning)

SEED = 42
N_JOBS = 5
np.random.seed(SEED)

RAIZ = Path.cwd() if (Path.cwd() / "data").exists() else Path.cwd().parent
DADOS = RAIZ / "data" / "working"
SAIDA = RAIZ / "outputs" / "baselines"
FIG = RAIZ / "outputs" / "figuras"
SAIDA.mkdir(parents=True, exist_ok=True)
FIG.mkdir(parents=True, exist_ok=True)

CATEGORICAS = ["Test Line", "Test Station", "Test Group", "Model Name",
               "Duty Type", "componente_grupo", "turno"]
NUMERICAS = ["hora_sin", "hora_cos", "dia_semana", "is_fim_semana"]
TARGET = "target"

print("Configuracao OK")
print("  RAIZ :", RAIZ)
print("  DADOS:", DADOS)
print("  SAIDA:", SAIDA)
print("  Semente:", SEED, "| n_jobs:", N_JOBS)

## Secao 1 - Carregamento dos dados

In [ ]:
treino = pd.read_parquet(DADOS / "treino.parquet")
val    = pd.read_parquet(DADOS / "val.parquet")
teste  = pd.read_parquet(DADOS / "teste.parquet")

print("Treino:", treino.shape)
print("Val   :", val.shape)
print("Teste :", teste.shape)
print()
print("Colunas:", list(treino.columns))
print()
print("Distribuicao do target (treino):")
print(treino[TARGET].value_counts().to_string())
print()
print("Categorias por feature categorica:")
for c in CATEGORICAS:
    print(f"  {c:22s} treino={treino[c].nunique():3d}  teste={teste[c].nunique():3d}")

print()
print("Categorias presentes no teste e ausentes no treino:")
for c in CATEGORICAS:
    novas = set(teste[c].unique()) - set(treino[c].unique())
    if novas:
        print(f"  {c}: {len(novas)} -> {sorted(novas)[:5]}")

## Secao 2 - Pre-processamento

`ColumnTransformer` com `OneHotEncoder(min_frequency=10, handle_unknown='ignore')` nas categoricas e `StandardScaler` nas numericas.

`min_frequency=10` agrupa categorias raras em `infrequent`, reduzindo esparsidade.

In [ ]:
preproc = ColumnTransformer(
    transformers=[
        ("cat", OneHotEncoder(min_frequency=10, handle_unknown="ignore",
                              sparse_output=True, dtype=np.float32), CATEGORICAS),
        ("num", StandardScaler(), NUMERICAS),
    ],
    remainder="drop",
)

X_tr = preproc.fit_transform(treino[CATEGORICAS + NUMERICAS])
X_va = preproc.transform(val[CATEGORICAS + NUMERICAS])
X_te = preproc.transform(teste[CATEGORICAS + NUMERICAS])

y_tr = treino[TARGET].values
y_va = val[TARGET].values
y_te = teste[TARGET].values

print("X_tr:", X_tr.shape, "| X_va:", X_va.shape, "| X_te:", X_te.shape)
print("y_tr:", y_tr.shape, "| classes:", len(np.unique(y_tr)))
print()
print("Features apos one-hot:", X_tr.shape[1])
print("Densidade:", round(X_tr.nnz / (X_tr.shape[0] * X_tr.shape[1]) * 100, 2), "%")

## Secao 3 - DummyClassifier (piso)

In [ ]:
def avaliar(nome, modelo, X, y, X_va, y_va, X_te, y_te):
    """Avalia um modelo treinado em treino/val/teste."""
    res = {"modelo": nome, "timestamp": datetime.now().isoformat(timespec="seconds")}
    labels = np.unique(np.concatenate([y_tr, y_va, y_te]))
    for split, Xs, ys in [("treino", X, y), ("val", X_va, y_va), ("teste", X_te, y_te)]:
        pred = modelo.predict(Xs)
        try:
            proba = modelo.predict_proba(Xs)
            top2 = top_k_accuracy_score(ys, proba, k=2, labels=labels)
        except Exception:
            top2 = float("nan")
        res[f"f1_macro_{split}"] = float(f1_score(ys, pred, average="macro", zero_division=0))
        res[f"f1_weighted_{split}"] = float(f1_score(ys, pred, average="weighted", zero_division=0))
        res[f"bal_acc_{split}"] = float(balanced_accuracy_score(ys, pred))
        res[f"acuracia_{split}"] = float(accuracy_score(ys, pred))
        res[f"top2_{split}"] = float(top2) if not np.isnan(top2) else None
    return res


resultados = []

t0 = time.time()
dummy = DummyClassifier(strategy="most_frequent", random_state=SEED)
dummy.fit(X_tr, y_tr)
res_dummy = avaliar("DummyClassifier", dummy, X_tr, y_tr, X_va, y_va, X_te, y_te)
res_dummy["tempo_treino_s"] = round(time.time() - t0, 2)
resultados.append(res_dummy)

print("DummyClassifier treinado em", res_dummy["tempo_treino_s"], "s")
print(f"  F1-macro teste: {res_dummy['f1_macro_teste']:.4f}")
print(f"  Bal. acc teste: {res_dummy['bal_acc_teste']:.4f}")
print(f"  Acuracia teste: {res_dummy['acuracia_teste']:.4f}")

with open(SAIDA / "03_dummy.json", "w") as f:
    json.dump(res_dummy, f, indent=2)
print("Checkpoint salvo: 03_dummy.json")

## Secao 4 - Regressao Logistica

Baseline linear com `class_weight='balanced'`.

In [ ]:
t0 = time.time()
logreg = LogisticRegression(
    max_iter=2000, class_weight="balanced",
    solver="lbfgs", n_jobs=N_JOBS, random_state=SEED,
)
logreg.fit(X_tr, y_tr)
res_lr = avaliar("LogisticRegression", logreg, X_tr, y_tr, X_va, y_va, X_te, y_te)
res_lr["tempo_treino_s"] = round(time.time() - t0, 2)
resultados.append(res_lr)

print("LogisticRegression treinado em", res_lr["tempo_treino_s"], "s")
print(f"  F1-macro teste: {res_lr['f1_macro_teste']:.4f}")
print(f"  Bal. acc teste: {res_lr['bal_acc_teste']:.4f}")
print()
print(classification_report(y_te, logreg.predict(X_te), zero_division=0))

with open(SAIDA / "04_logreg.json", "w") as f:
    json.dump(res_lr, f, indent=2)
with open(SAIDA / "04_logreg.pkl", "wb") as f:
    pickle.dump(logreg, f)
print("Checkpoint salvo: 04_logreg.json + .pkl")

## Secao 5 - Random Forest

300 arvores, profundidade 15, `class_weight='balanced_subsample'`.

In [ ]:
t0 = time.time()
rf = RandomForestClassifier(
    n_estimators=300, max_depth=15, min_samples_leaf=5,
    class_weight="balanced_subsample",
    n_jobs=N_JOBS, random_state=SEED,
)
rf.fit(X_tr, y_tr)
res_rf = avaliar("RandomForest", rf, X_tr, y_tr, X_va, y_va, X_te, y_te)
res_rf["tempo_treino_s"] = round(time.time() - t0, 2)
resultados.append(res_rf)

print("RandomForest treinado em", res_rf["tempo_treino_s"], "s")
print(f"  F1-macro teste: {res_rf['f1_macro_teste']:.4f}")
print(f"  Bal. acc teste: {res_rf['bal_acc_teste']:.4f}")
print()
print(classification_report(y_te, rf.predict(X_te), zero_division=0))

with open(SAIDA / "05_randomforest.json", "w") as f:
    json.dump(res_rf, f, indent=2)
with open(SAIDA / "05_randomforest.pkl", "wb") as f:
    pickle.dump(rf, f)
print("Checkpoint salvo: 05_randomforest.json + .pkl")

## Secao 6 - HistGradientBoosting

Gradient boosting com early stopping e `class_weight='balanced'`.

In [ ]:
t0 = time.time()

# HistGradientBoosting exige matriz densa (nao aceita esparso)
X_tr_d = X_tr.toarray() if hasattr(X_tr, "toarray") else X_tr
X_va_d = X_va.toarray() if hasattr(X_va, "toarray") else X_va
X_te_d = X_te.toarray() if hasattr(X_te, "toarray") else X_te
print("Matrizes densificadas:", X_tr_d.shape, "| memoria aprox:",
      round(X_tr_d.nbytes / 1e6, 1), "MB")

hgb = HistGradientBoostingClassifier(
    max_iter=300, max_depth=10, learning_rate=0.08,
    min_samples_leaf=20, l2_regularization=0.1,
    early_stopping=True, validation_fraction=0.15,
    n_iter_no_change=20, class_weight="balanced",
    random_state=SEED,
)
hgb.fit(X_tr_d, y_tr)
res_hgb = avaliar("HistGradientBoosting", hgb, X_tr_d, y_tr, X_va_d, y_va, X_te_d, y_te)
res_hgb["tempo_treino_s"] = round(time.time() - t0, 2)
res_hgb["n_iter_real"] = int(hgb.n_iter_)
resultados.append(res_hgb)

print("HistGradientBoosting treinado em", res_hgb["tempo_treino_s"], "s")
print(f"  Iteracoes reais: {res_hgb['n_iter_real']} (early stopping)")
print(f"  F1-macro teste: {res_hgb['f1_macro_teste']:.4f}")
print(f"  Bal. acc teste: {res_hgb['bal_acc_teste']:.4f}")
print()
print(classification_report(y_te, hgb.predict(X_te_d), zero_division=0))

with open(SAIDA / "06_hgb.json", "w") as f:
    json.dump(res_hgb, f, indent=2)
with open(SAIDA / "06_hgb.pkl", "wb") as f:
    pickle.dump(hgb, f)
print("Checkpoint salvo: 06_hgb.json + .pkl")

## Secao 7 - Tabela comparativa

In [ ]:
df_res = pd.DataFrame(resultados)
cols = ["modelo",
        "f1_macro_treino", "f1_macro_val", "f1_macro_teste",
        "f1_weighted_teste", "bal_acc_teste", "acuracia_teste",
        "top2_teste", "tempo_treino_s"]
df_res = df_res[[c for c in cols if c in df_res.columns]]

print("=" * 90)
print("TABELA COMPARATIVA - conjunto de TESTE")
print("=" * 90)
print(df_res.to_string(index=False))
print()
df_res.to_csv(SAIDA / "07_tabela_comparativa.csv", index=False)
print("Salvo: 07_tabela_comparativa.csv")

pos = np.arange(len(df_res))
larg = 0.35
fig, ax = plt.subplots(figsize=(9, 4.5))
ax.bar(pos - larg/2, df_res["f1_macro_teste"], larg, label="F1-macro", color="#2E86AB")
ax.bar(pos + larg/2, df_res["bal_acc_teste"], larg, label="Balanced accuracy", color="#A23B72")
ax.set_xticks(pos)
ax.set_xticklabels(df_res["modelo"], rotation=15, ha="right")
ax.set_ylim(0, 1)
ax.set_ylabel("Score")
ax.set_title("Baselines - desempenho no conjunto de teste")
ax.legend()
ax.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.savefig(FIG / "07_comparativo_baselines.png", dpi=150)
plt.close()
print("Figura salva: 07_comparativo_baselines.png")

## Secao 8 - Matrizes de confusao

In [ ]:
classes = sorted(np.unique(y_tr))

# cada modelo precisa da matriz no formato em que foi treinado
modelos = {
    "Dummy":        (dummy, X_te),
    "LogReg":       (logreg, X_te),
    "RandomForest": (rf, X_te),
    "HGB":          (hgb, X_te_d),
}

fig, axes = plt.subplots(2, 2, figsize=(13, 11))
for ax, (nome, (modelo, X_pred)) in zip(axes.ravel(), modelos.items()):
    pred = modelo.predict(X_pred)
    cm = confusion_matrix(y_te, pred, labels=classes)
    cm_n = cm.astype(float) / cm.sum(axis=1, keepdims=True)
    im = ax.imshow(cm_n, cmap="Blues", vmin=0, vmax=1)
    ax.set_xticks(range(len(classes)))
    ax.set_yticks(range(len(classes)))
    rotulos = [c.replace(" ", "\n") for c in classes]
    ax.set_xticklabels(rotulos, rotation=45, ha="right", fontsize=8)
    ax.set_yticklabels(rotulos, fontsize=8)
    ax.set_title(nome, fontsize=11)
    ax.set_xlabel("Predito")
    ax.set_ylabel("Real")
    for i in range(len(classes)):
        for j in range(len(classes)):
            v = cm_n[i, j]
            if v > 0.01:
                ax.text(j, i, f"{v:.2f}", ha="center", va="center",
                        color="white" if v > 0.5 else "black", fontsize=7)

    # salva a matriz numerica para a analise de erro do artigo
    np.savetxt(SAIDA / f"08_cm_{nome}.csv", cm, delimiter=",", fmt="%d")

plt.colorbar(im, ax=axes, fraction=0.03, pad=0.02, label="Proporcao por classe real")
plt.suptitle("Matrizes de confusao normalizadas - conjunto de teste", y=0.995)
plt.savefig(FIG / "08_matrizes_confusao.png", dpi=150, bbox_inches="tight")
plt.close()
print("Figura salva: 08_matrizes_confusao.png")
print("Matrizes numericas salvas: 08_cm_<modelo>.csv")

# resumo textual por classe (material para a analise de erro)
print()
print("Recall por classe no melhor modelo:")
melhor_nome = max(resultados[1:], key=lambda r: r["f1_macro_teste"])["modelo"]
melhor_modelo, melhor_X = modelos[
    {"LogisticRegression": "LogReg",
     "RandomForest": "RandomForest",
     "HistGradientBoosting": "HGB"}[melhor_nome]
]
rel = classification_report(y_te, melhor_modelo.predict(melhor_X),
                            zero_division=0, output_dict=True)
for cls in classes:
    if cls in rel:
        print(f"  {cls:30s} precision={rel[cls]['precision']:.3f}  "
              f"recall={rel[cls]['recall']:.3f}  f1={rel[cls]['f1-score']:.3f}  n={int(rel[cls]['support'])}")

## Secao 9 - Importancia das features

In [ ]:
melhor = max(resultados[1:], key=lambda r: r["f1_macro_teste"])["modelo"]
mapa_modelos = {"LogisticRegression": logreg, "RandomForest": rf, "HistGradientBoosting": hgb}
modelo_escolhido = mapa_modelos[melhor]
print("Modelo escolhido para importance:", melhor)

nomes_completos = (list(preproc.named_transformers_["cat"].get_feature_names_out(CATEGORICAS))
                   + NUMERICAS)

def grupo_de(nome):
    for c in CATEGORICAS:
        if nome.startswith(c + "_"):
            return c
    return nome

# permutation_importance precisa do mesmo formato usado no fit
X_te_imp = X_te
if melhor == "HistGradientBoosting":
    X_te_imp = X_te.toarray() if hasattr(X_te, "toarray") else X_te

r = permutation_importance(modelo_escolhido, X_te_imp, y_te, n_repeats=10,
        scoring="f1_macro", n_jobs=N_JOBS, random_state=SEED)

imp = pd.DataFrame({
    "feature": nomes_completos,
    "importancia_media": r.importances_mean,
    "importancia_std": r.importances_std,
}).sort_values("importancia_media", ascending=False)

imp["grupo"] = imp["feature"].map(grupo_de)
imp_grp = imp.groupby("grupo")["importancia_media"].sum().sort_values(ascending=False)

print()
print("Importancia agregada por variavel original (queda no F1-macro):")
print(imp_grp.to_string())
print()

imp_grp.to_csv(SAIDA / "09_importance_por_grupo.csv", header=["importancia"])

fig, ax = plt.subplots(figsize=(8, 4))
imp_grp.plot(kind="barh", ax=ax, color="#2E86AB")
ax.invert_yaxis()
ax.set_xlabel("Queda media no F1-macro")
ax.set_title(f"Permutation importance - {melhor} (teste)")
ax.grid(axis="x", alpha=0.3)
plt.tight_layout()
plt.savefig(FIG / "09_importance.png", dpi=150)
plt.close()
print("Salvo: 09_importance_por_grupo.csv + 09_importance.png")

## Secao 10 - Consolidacao final

In [ ]:
todos = {}
for arq in sorted(SAIDA.glob("*.json")):
    if "consolidado" in arq.name:
        continue
    with open(arq) as f:
        todos[arq.stem] = json.load(f)

with open(SAIDA / "10_resultados_consolidados.json", "w") as f:
    json.dump(todos, f, indent=2, ensure_ascii=False)

print("=" * 90)
print("RESUMO FINAL - F1-macro no conjunto de TESTE")
print("=" * 90)
for k, v in sorted(todos.items(), key=lambda x: -x[1]["f1_macro_teste"]):
    print(f"  {v['modelo']:24s}  F1-macro={v['f1_macro_teste']:.4f}  "
          f"bal.acc={v['bal_acc_teste']:.4f}  tempo={v['tempo_treino_s']}s")

print()
print("Arquivos gerados em", SAIDA)
for f in sorted(SAIDA.iterdir()):
    print("  ", f.name)
print()
print("Figuras em", FIG)
for f in sorted(FIG.iterdir()):
    print("  ", f.name)
print()
print("PROXIMO PASSO: comparar estes numeros com a MLP do notebook 06.")